# ChromaNeRV CVPR Phase 2
Run preflight first. Training and latency are opt-in. Use an A100 runtime for the latency benchmark.


In [ ]:
RUN_TRAINING = False
RUN_LATENCY = False
ONLY_SEQUENCE = 'HoneyBee'
ONLY_VARIANT = 'rgbsplit_yc_a320'
MAX_RUNS = 1
EPOCHS = 150
BUNNY_ZIP = ''  # Set to the original 132-frame Bunny ZIP in MyDrive
BUNNY_COMMAND = ''  # Set to the original Bunny checkpoint command.txt
LATENCY_MANIFEST = ''  # JSON with six HoneyBee checkpoint/config/command SHA-256 records


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, zipfile, hashlib, shutil, json
drive_root = Path('/content/drive/MyDrive')
persistent = drive_root / 'ChromaNeRV_CVPR_Phase2'
persistent.mkdir(parents=True, exist_ok=True)
local = Path('/content/chromanerv_phase2')
local.mkdir(exist_ok=True)
repo = Path('/content/ChromaHNeRV')
if not repo.exists(): subprocess.run(['git','clone','https://github.com/HannanAkhtar/ChromaHNeRV.git',str(repo)], check=True)
subprocess.run(['git','-C',str(repo),'fetch','origin','experiment/cvpr27-phase2-controls-latency'], check=True)
subprocess.run(['git','-C',str(repo),'checkout','experiment/cvpr27-phase2-controls-latency'], check=True)
sha = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'], text=True).strip()
print('Repository commit:', sha)
subprocess.run([sys.executable,'-m','pip','install','-r',str(repo/'requirements_modern.txt')], check=True)


In [ ]:
def locate_zip(name):
    found = list(drive_root.rglob(name))
    if len(found) != 1: raise FileNotFoundError(f'Expected one {name} in MyDrive, found {len(found)}')
    return found[0]
def extract_verified(path, destination):
    local_zip = local / path.name
    if not local_zip.exists() or local_zip.stat().st_size != path.stat().st_size: shutil.copy2(path, local_zip)
    print(path.name, local_zip.stat().st_size, hashlib.sha256(local_zip.read_bytes()).hexdigest())
    with zipfile.ZipFile(local_zip) as archive:
        bad = archive.testzip()
        if bad: raise ValueError(f'Corrupt ZIP member: {bad}')
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination.resolve()): raise ValueError('Unsafe ZIP path')
        archive.extractall(destination)
    return destination
uvg_root = extract_verified(locate_zip('ChromaNeRV_UVG132.zip'), local/'uvg_archive')
checkpoint_root = extract_verified(locate_zip('ChromaNeRV_CVPR_Checkpoints_20261115_021417.zip'), local/'checkpoints_archive')
print('UVG:', uvg_root, 'Checkpoints:', checkpoint_root)


In [ ]:
subprocess.run([sys.executable,'-m','pytest','-q',str(repo/'tests')], cwd=repo, check=True)
def find_uvg_root(root):
    candidates = [p for p in root.rglob('HoneyBee') if p.is_dir() and len(list(p.glob('*.png'))) == 132]
    if len(candidates) != 1: raise ValueError(f'Expected one 132-frame HoneyBee directory, found {candidates}')
    return candidates[0].parent
uvg_frames = find_uvg_root(uvg_root)
bunny_frames = ''
if BUNNY_ZIP: bunny_frames = str(extract_verified(Path(BUNNY_ZIP), local/'bunny_archive'))
base = [sys.executable,str(repo/'run_cvpr27_phase2.py'),'--uvg-root',str(uvg_frames),'--output-root',str(local/'runs'),'--backup-root',str(persistent/'runs')]
subprocess.run(base + ['--dry-run'], cwd=repo, check=True)


In [ ]:
if RUN_TRAINING:
    command = base + ['--max-runs',str(MAX_RUNS),'--epochs',str(EPOCHS),'--strict-backup']
    if ONLY_SEQUENCE: command += ['--only-sequence',ONLY_SEQUENCE]
    if ONLY_VARIANT: command += ['--only-variant',ONLY_VARIANT]
    if ONLY_SEQUENCE == 'Bunny': command += ['--bunny-root',bunny_frames,'--bunny-command',BUNNY_COMMAND]
    subprocess.run(command, cwd=repo, check=True)
else: print('Training disabled. Set RUN_TRAINING=True after preflight.')


In [ ]:
def discover_latency_manifest(root):
    from benchmark_decoder_latency import ARCHITECTURES
    records = {}
    for checkpoint in root.rglob('epoch150.pth'):
        config, command = checkpoint.parent/'config.json', checkpoint.parent/'command.txt'
        if not config.is_file() or not command.is_file(): continue
        cfg = json.loads(config.read_text())
        if str(cfg.get('vid','')).lower() != 'honeybee' or float(cfg.get('modelsize',0)) != 1.5: continue
        experiment = cfg.get('experiment','')
        key = experiment if experiment == 'rgb444_hnerv' else f"{experiment}_w{cfg.get('branch_width')}"
        if key not in ARCHITECTURES: continue
        hashes = {field: hashlib.sha256(path.read_bytes()).hexdigest() for field,path in [('checkpoint',checkpoint),('config',config),('command',command)]}
        if key in records: raise ValueError(f'Duplicate latency checkpoint: {key}')
        records[key] = {'checkpoint':str(checkpoint),'config':str(config),'command':str(command),'sha256':hashes}
    missing = set(ARCHITECTURES) - set(records)
    if missing: raise FileNotFoundError(f'Missing archived HoneyBee checkpoints: {sorted(missing)}; locate additional originals in persistent archive')
    destination = local/'phase2_latency_manifest.json'
    destination.write_text(json.dumps(records,indent=2))
    return destination
if RUN_LATENCY:
    sys.path.insert(0,str(repo))
    manifest_path = LATENCY_MANIFEST or str(discover_latency_manifest(checkpoint_root))
    subprocess.run([sys.executable,str(repo/'benchmark_decoder_latency.py'),'--manifest',manifest_path,'--frames',str(uvg_frames/'HoneyBee'),'--output',str(persistent/'latency')],cwd=repo,check=True)
else: print('Latency disabled. Set RUN_LATENCY=True on an A100 after supplying six original checkpoints.')
